<a href="https://colab.research.google.com/github/Nefeli-Apostolou/GM-Project---Fraud-Detection/blob/main/Ethereum_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-cloud-bigquery google-auth google-auth-oauthlib

from google.colab import auth
auth.authenticate_user()

In [ ]:
import os
from google.cloud import bigquery

PROJECT_ID = ""   # <-- your actual project id
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
client = bigquery.Client(project=PROJECT_ID)

print("Connected to project:", client.project)


Connected to project: hidden-lyceum-477816-s7


In [ ]:
sql = """
SELECT
  block_number,
  `hash`,
  from_address,
  to_address,
  value,
  gas,
  gas_price,
  receipt_gas_used,
  block_timestamp
FROM `bigquery-public-data.crypto_ethereum.transactions`
WHERE block_timestamp BETWEEN
  TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 DAY)
  AND TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
ORDER BY block_timestamp DESC
LIMIT 1000000000
"""

job = client.query(sql)
df = job.to_dataframe()
print("Rows:", len(df))
df.head()

KeyboardInterrupt: 

In [ ]:
df.to_csv("eth_tx_lot_days.csv", index=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df.to_csv("/content/drive/MyDrive/eth_tx_lot_days.csv", index=False)

Mounted at /content/drive


-------------------------------------------------------
# This section is only relevant for the DDMCS Project

In [ ]:
# Loop over 5 rolling 2-day windows
for i in range(5):
    # define offsets in days (larger offset = further in the past)
    start_offset = 3 + 2 * i   # older boundary
    end_offset   = 1 + 2 * i   # newer boundary (up to yesterday for i = 0)

    sql = f"""
    SELECT
      block_number,
      `hash`,
      from_address,
      to_address,
      value,
      gas,
      gas_price,
      receipt_gas_used,
      block_timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE block_timestamp BETWEEN
      TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {start_offset} DAY)
      AND TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {end_offset} DAY)
    ORDER BY block_timestamp DESC
    LIMIT 10000000
    """

    job = client.query(sql)
    df = job.to_dataframe()
    print(f"Window {i+1}: {start_offset}–{end_offset} days ago, rows = {len(df)}")

    # save one CSV per window
    fname = f"eth_tx_{start_offset}to{end_offset}_days_ago.csv"
    df.to_csv(fname, index=False)
    print("Saved:", fname)


Window 1: 3–1 days ago, rows = 3002718
Saved: eth_tx_3to1_days_ago.csv
Window 2: 5–3 days ago, rows = 2885632
Saved: eth_tx_5to3_days_ago.csv
Window 3: 7–5 days ago, rows = 3089067
Saved: eth_tx_7to5_days_ago.csv
Window 4: 9–7 days ago, rows = 3017515
Saved: eth_tx_9to7_days_ago.csv
Window 5: 11–9 days ago, rows = 2972652
Saved: eth_tx_11to9_days_ago.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Directory in your Google Drive where CSVs will be stored
OUT_DIR = "/content/drive/MyDrive/eth_windows/"
os.makedirs(OUT_DIR, exist_ok=True)

for i in range(5):
    start_offset = 3 + 2 * i   # older boundary
    end_offset   = 1 + 2 * i   # newer boundary

    sql = f"""
    SELECT
      block_number,
      `hash`,
      from_address,
      to_address,
      value,
      gas,
      gas_price,
      receipt_gas_used,
      block_timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE block_timestamp BETWEEN
      TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {start_offset} DAY)
      AND TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {end_offset} DAY)
    ORDER BY block_timestamp DESC
    LIMIT 10000000
    """

    job = client.query(sql)
    df = job.to_dataframe()

    print(f"Window {i+1}: {start_offset}–{end_offset} days ago → rows = {len(df)}")

    # Path inside Google Drive
    fname = f"eth_tx_{start_offset}to{end_offset}_days_ago.csv"
    full_path = OUT_DIR + fname

    # Save to Drive
    df.to_csv(full_path, index=False)
    print("Saved to:", full_path)


In [ ]:
import pandas as pd
import numpy as np

# --- ASSUMPTION: Load your dataframe here ---
# Replace 'your_file.csv' with the name of the file you downloaded from Dune/AWS.
file_name = "eth_tx_7to5_days_ago.csv"
df = pd.read_csv(file_name)

# --- CORE LOGIC ---

# 1. Define the columns based on your screenshot
# The columns are 'from_address' and 'to_address'
from_col = 'from_address'
to_col = 'to_address'

# 2. Concatenate the two columns into a single Series
all_addresses_series = pd.concat([df[from_col], df[to_col]])

# 3. Calculate the number of unique addresses (nodes)
unique_address_count = all_addresses_series.nunique()

# --- OUTPUT ---
print(f"Total Transactions (Edges): {len(df):,}")
print(f"Total Unique Addresses (Nodes): {unique_address_count:,}")

Total Transactions (Edges): 3,089,067
Total Unique Addresses (Nodes): 983,930


In [ ]:
# Force conversion — coercing errors if needed
df['block_timestamp'] = pd.to_datetime(df['block_timestamp'], errors='coerce', utc=True)

# Check if any failed to convert
print(df['block_timestamp'].isna().sum(), "rows could not be parsed")

# Now compute unique days and hours
unique_days = df['block_timestamp'].dt.date.nunique()
unique_hours = df['block_timestamp'].dt.floor('H').nunique()

print("Unique days:", unique_days)
print("Unique hours:", unique_hours)

df.head()  # Display the first few rows of the dataframe for verification



0 rows could not be parsed
Unique days: 3
Unique hours: 49


/tmp/ipython-input-2297310593.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  unique_hours = df['block_timestamp'].dt.floor('H').nunique()


,block_number,hash,from_address,to_address,value,gas,gas_price,receipt_gas_used,block_timestamp
0,23798450,0xcf10320ede43557a4c1d5467aed2e35cd384d6ef6817...,0x509b2c13a2c5534dc70d6cd589c0c7420c9bd6c5,0xdac17f958d2ee523a2206206994597c13d831ec7,0.000000e+00,48936,2297241520,48549,2025-11-14 15:53:23+00:00
1,23798450,0x1abec426fae093a74dc30d9bd77e647ede99db590af6...,0x0172f4cd7a971cec5da4042453e02fde3ff58410,0x7da2641000cbb407c329310c461b2cb9c70c3046,0.000000e+00,53355,1397241519,46212,2025-11-14 15:53:23+00:00
2,23798450,0x73c31c3f55b65f0c5469018c99bc6ab2054e64ffa078...,0x3d011bbbceef340d8c9c0f44ef7b84a06da1fc9c,0x13e6e1833c4891e3c36084ffd250c8fb5ca7cff6,1.000000e+16,100000,2297241519,21000,2025-11-14 15:53:23+00:00
3,23798450,0x1d24e4e64c6e5c0cab6edc801356966e5ae978f9983b...,0xbbad96336943a36e3e292d973b3382dd0ba4d1ef,0xa69babef1ca67a37ffaf7a485dfff3382056e78c,3.299390e+06,275844,1297241519,117922,2025-11-14 15:53:23+00:00
4,23798450,0x4599139517b5424c0edcc69afc537bedea05e6932b28...,0xa2527ce28c4172825850cb637bb1902ba4637d61,0xdac17f958d2ee523a2206206994597c13d831ec7,0.000000e+00,210000,1307048029,41297,2025-11-14 15:53:23+00:00
